# CodeOrbit on SWE-bench - Kaggle (unattended GPU run)

Runs `scripts/swebench.py --embed` on a Kaggle GPU **in the background** - no
browser tab needs to stay open.

### Once, in the right-hand panel (Settings)
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On - needs a phone-verified Kaggle account

### To run
1. Set `SPLIT`, `MODE` and `ARMS` in the first code cell (default: all 300
   SWE-bench Multilingual tasks, hybrid vs hybrid-graph vs fusion).
2. **Save Version -> Save & Run All (Commit) -> Save.** You can close the browser.
3. Later: open the notebook -> **Output** tab -> download `codeorbit-runs.zip`.
   The log of the run is in the version's **Logs**.

### If a run stops early
Kaggle allows 12 h; the run stops itself after `MAX_HOURS` and keeps everything
finished so far. To continue: **Add Input -> Your Work -> this notebook** (its
latest version), copy the path of the run folder it shows under
`/kaggle/input/.../codeorbit-runs/`, paste it into `RESUME_FROM`, and
Save & Run All again. Finished tasks are skipped.

## Settings

In [ ]:
SPLIT = 'multilingual'  # 'multilingual' - 300 tasks in Go, Rust, Java, C/C++, Ruby, PHP, JS, TS
                        # 'verified'     - SWE-bench Verified, 500 Python tasks (93 also in Lite)
                        # 'lite'         - SWE-bench Lite, 300 Python tasks
MODE = 'full'      # 'full'    - every task of SPLIT
                   # 'pilot'   - 6 small tasks, ~15 min: check everything works first
                   # 'dev'     - Lite only: the 60-task development sample
                   # 'heldout' - Lite only: the other 240 tasks
ARMS = 'hybrid,hybrid-graph,fusion'   # the baseline, + CodeOrbit's graph context, + its ranking
MAX_HOURS = 11     # stop cleanly before Kaggle's 12 h limit
RESUME_FROM = ''   # to continue a stopped run: '/kaggle/input/.../codeorbit-runs/<run folder>'

## 1. GPU check, get CodeOrbit

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv

RUNS = '/kaggle/working/codeorbit-runs'   # saved as the notebook's output
CHECKOUTS = '/tmp/checkouts'              # several GB of repositories: not output
SRC = '/tmp/codeorbit'
os.makedirs(RUNS, exist_ok=True)

%cd /tmp
!rm -rf {SRC}
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git {SRC}
%cd {SRC}
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 2. Start Ollama on the GPU

The last line should report the model on **GPU**. If it says CPU, check the
Accelerator setting.

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull nomic-embed-text

t = time.perf_counter()
r = requests.post('http://127.0.0.1:11434/api/embed', timeout=300,
                  json={'model': 'nomic-embed-text',
                        'input': [f'def f{i}(x): return x' for i in range(256)]})
r.raise_for_status()
print(f'256 embeddings in {time.perf_counter() - t:.1f}s')
!ollama ps

## 3. Download the tasks

In [ ]:
!python scripts/swebench.py download --split {SPLIT}

## 4. Run

Selection per `MODE`. Every finished task is written to `RUNS` at once, so even
a run that is cut off keeps its results.

In [ ]:
import shutil
selection = {
    'full':    '',
    'pilot':   '--repos gin-gonic/gin,axios/axios' if SPLIT == 'multilingual'
               else '--repos psf/requests',
    'dev':     '--sample 60 --seed 0',
    'heldout': '--exclude-sample 60 --exclude-seed 0',
}[MODE]

resume = ''
if RESUME_FROM:
    # /kaggle/input is read-only: continue in a copy under the output folder.
    dst = os.path.join(RUNS, os.path.basename(RESUME_FROM.rstrip('/')))
    if not os.path.exists(dst):
        shutil.copytree(RESUME_FROM, dst)
    resume = f'--resume {dst}'
    print('resuming', dst)

# Built in Python first: a missing value is a clear NameError, not a garbled command.
cmd = (f'python scripts/swebench.py run --split {SPLIT} {selection} --arms {ARMS} --embed '
       f'--tokenizer hf:microsoft/Phi-4-mini-instruct --checkouts {CHECKOUTS} '
       f'--out {RUNS} --max-hours {MAX_HOURS} {resume}')
print(cmd)
!{cmd}

## 5. Results and download

In [ ]:
import glob, json
done = [r for r in sorted(glob.glob(f'{RUNS}/*-swebench-*'))
        if os.path.exists(f'{r}/meta.json')
        and json.load(open(f'{r}/meta.json'))['run'].get('complete', True)]
print('\n'.join(done) or 'no finished run yet')
if done:
    # The latest run: table, then head-to-head per task
    !python scripts/bench_report.py {done[-1]} \
        --versus fusion,hybrid --versus hybrid-graph,hybrid --versus fusion,hybrid-graph

# One file to download from the Output tab
shutil.make_archive('/kaggle/working/codeorbit-runs', 'zip', RUNS)
print('download /kaggle/working/codeorbit-runs.zip from the Output tab')